In [821]:
import pandas as pd
import json

In [822]:
with open('../election-stations-2569.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# หาหน่วยเลือกตั้งของจังหวัด
rayong = next((p for p in data['provinces'] if p['name'] == 'ระยอง'), None)
# print(f"ระยอง {rayong['total_stations']} หน่วยเลือกตั้ง")
d = dict()
sd2d = dict()
t = 0
# หาหน่วยเลือกตั้งจากรหัส
def find_total_station(area,province):
    for p in data['provinces']:
        if p['name'] == province:
            for a in p['areas']:
                if a['area'] == area:
                    for unit in a['stations']:
                        if unit['district'] not in d:
                            d[unit['district']] = dict()
                        if unit['subdistrict'] not in  d[unit['district']]:
                            d[unit['district']][unit['subdistrict']] = 1
                            sd2d[unit['subdistrict']] = unit['district']
                        else:
                            d[unit['district']][unit['subdistrict']] += 1
    return None

find_total_station(2, 'ลำปาง')
print(d)
print(sd2d)
for k, v in d.items():
    for e, g in v.items():
        t+=g
print(t)

{'เมืองลำปาง': {'บ้านแลง': 12, 'บ้านเสด็จ': 19}, 'งาว': {'หลวงเหนือ': 7, 'หลวงใต้': 8, 'บ้านโป่ง': 12, 'บ้านร้อง': 13, 'ปงเตา': 13, 'นาแก': 6, 'บ้านอ้อน': 8, 'บ้านแหง': 9, 'บ้านหวด': 7, 'แม่ตีบ': 7}, 'แจ้ห่ม': {'แจ้ห่ม': 14, 'บ้านสา': 10, 'ปงดอน': 9, 'แม่สุก': 12, 'เมืองมาย': 7, 'ทุ่งผึ้ง': 8, 'วิเชตนคร': 12}, 'วังเหนือ': {'ทุ่งฮั้ว': 12, 'วังเหนือ': 12, 'วังใต้': 7, 'ร่องเคาะ': 18, 'วังทอง': 9, 'วังซ้าย': 10, 'วังแก้ว': 7, 'วังทรายคำ': 9}, 'เมืองปาน': {'เมืองปาน': 11, 'บ้านขอ': 14, 'ทุ่งกว๋าว': 16, 'แจ้ซ้อน': 15, 'หัวเมือง': 9}}
{'บ้านแลง': 'เมืองลำปาง', 'บ้านเสด็จ': 'เมืองลำปาง', 'หลวงเหนือ': 'งาว', 'หลวงใต้': 'งาว', 'บ้านโป่ง': 'งาว', 'บ้านร้อง': 'งาว', 'ปงเตา': 'งาว', 'นาแก': 'งาว', 'บ้านอ้อน': 'งาว', 'บ้านแหง': 'งาว', 'บ้านหวด': 'งาว', 'แม่ตีบ': 'งาว', 'แจ้ห่ม': 'แจ้ห่ม', 'บ้านสา': 'แจ้ห่ม', 'ปงดอน': 'แจ้ห่ม', 'แม่สุก': 'แจ้ห่ม', 'เมืองมาย': 'แจ้ห่ม', 'ทุ่งผึ้ง': 'แจ้ห่ม', 'วิเชตนคร': 'แจ้ห่ม', 'ทุ่งฮั้ว': 'วังเหนือ', 'วังเหนือ': 'วังเหนือ', 'วังใต้': 'วังเหนือ', 'ร่องเคาะ': 'วังเ

In [823]:
d['นอกเขต'] = dict()
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    sd2d[sub_dist] = 'นอกเขต'
    d['นอกเขต'][sub_dist] = 1
    # all_subdistrict.append(sub_dist)

In [824]:
d['วังเหนือ']['บ้านใหม่'] = 4
d['วังเหนือ']['วังเหนือ'] = 8
d['แจ้ห่ม']['แจ้ห่ม(ในเขต)'] = 6
d['แจ้ห่ม']['แจ้ห่ม(นอกเขต)'] = 8
d['แจ้ห่ม'].pop('แจ้ห่ม', None)
sd2d['บ้านใหม่'] = 'วังเหนือ'
sd2d['แจ้ห่ม(ในเขต)'] = "แจ้ห่ม"
sd2d['แจ้ห่ม(นอกเขต)'] = "แจ้ห่ม"

In [825]:
all_distrcit = []
all_subdistrict = []
district_count = {}
for k,v in d.items():
    all_distrcit.append(k)
    all_subdistrict += v.keys()
    for key in v.keys():
        if k not in district_count:
            district_count[k] = v[key]
        else: district_count[k]+= v[key]

In [826]:
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    all_subdistrict.append(sub_dist)

In [827]:
df = pd.read_csv("../ocr-result1/master_results.csv")

In [828]:
wrong_districts = []
for wrong_district in df['district'].unique().tolist():
    if wrong_district not in all_distrcit and wrong_district != 'นอกเขต':
        wrong_districts.append(wrong_district)

wrong_districts

[]

In [829]:
from rapidfuzz import process
def automate_edit(feature, check, wrongs, df):
    wrong2correct = dict()
    scores = dict()
    lis = []
    for text in wrongs:
        match = process.extractOne(text, check, score_cutoff=50)

        if match:
            result, score, index = match
            wrong2correct[text] = result
            scores[text] = round(score,2)
            if scores[text] >= 70:
                df.loc[df[feature] == text, feature] = result
            else:
                lis.append(text)
            print(f"OCR: '{text}' -> Corrected: '{result}' (Confidence: {score:.2f}%)")
        else:
            wrong2correct[text] = "can't find"
            scores[text] = 0
            lis.append(text)
            print(f"OCR: '{text}' -> [Manual Review Needed]")
    return wrong2correct, scores, lis

In [830]:
df_test = df.copy()
wrong2correct, scores, list_wrongs = automate_edit('district', all_distrcit, wrong_districts, df_test)

In [831]:
wrong_sds = []
for wrong_sd in df_test['sub-district'].unique().tolist():
    if wrong_sd not in all_subdistrict:
        wrong_sds.append(wrong_sd)

wrong_sds

[]

In [832]:
df_test.loc[df_test['sub-district'] == 'เเจ้ซ้อน', 'sub-district'] = 'แจ้ซ้อน'

In [833]:
l = df_test['sub-district'].unique().tolist()
for sd in all_subdistrict:
    if sd not in l:
        print(f"เขต {sd} หายไปทั้ง บช และ เขต")

In [834]:
df = df_test.copy()

In [835]:
df['district'].unique().tolist()

['นอกเขต', 'งาว', 'วังเหนือ', 'เมืองปาน', 'เมืองลำปาง', 'แจ้ห่ม']

In [836]:
correct_names = [
    "นางสาววิชุดา ว่องวัฒนวิโรจน์",
    "นางสาวสุวิภา กุศลจูง",
    "นายศรีพรหม หอมยก",
    "นายสมบูรณ์ รูปสะอาด",
    "นายดาชัย เอกปฐพี",
    "นายธนาธร โล่ห์สุนทร",
    "พันเอกสันทัด ภัทรกิตตินนท์",
]

In [837]:
party2number = dict()
with open("../partys_lists_number.txt", 'r', encoding="utf-8") as f:
    lines = []
    for line in f:
        if line.strip() == "":continue
        num, party = line.strip().split(":")
        num = int(num.strip().split()[1])
        party = party.strip().removeprefix("พรรค")
        lines.append(party)
        party2number[party] = num
for i in range(1,8):
    party2number[correct_names[i-1]] = i
correct_names += lines

In [838]:
correct_names

['นางสาววิชุดา ว่องวัฒนวิโรจน์',
 'นางสาวสุวิภา กุศลจูง',
 'นายศรีพรหม หอมยก',
 'นายสมบูรณ์ รูปสะอาด',
 'นายดาชัย เอกปฐพี',
 'นายธนาธร โล่ห์สุนทร',
 'พันเอกสันทัด ภัทรกิตตินนท์',
 'ไทยทรัพย์ทวี',
 'เพื่อชาติไทย',
 'ใหม่',
 'มิติใหม่',
 'รวมใจไทย',
 'รวมไทยสร้างชาติ',
 'พลวัต',
 'ประชาธิปไตยใหม่',
 'เพื่อไทย',
 'ทางเลือกใหม่',
 'เศรษฐกิจ',
 'เสรีรวมไทย',
 'รวมพลังประชาชน',
 'ท้องที่ไทย',
 'อนาคตไทย',
 'พลังเพื่อไทย',
 'ไทยชนะ',
 'พลังสังคมใหม่',
 'สังคมประชาธิปไตย',
 'ฟิวชัน',
 'ไทรวมพลัง',
 'ก้าวอิสระ',
 'ปวงชนไทย',
 'วิชชั่นใหม่',
 'เพื่อชีวิตใหม่',
 'คลองไทย',
 'ประชาธิปัตย์',
 'ไทยก้าวหน้า',
 'ไทยภักดี',
 'แรงงานสร้างชาติ',
 'ประชากรไทย',
 'ครูไทยเพื่อประชาชน',
 'ประชาชาติ',
 'สร้างอนาคตไทย',
 'รักชาติ',
 'ไทยพร้อม',
 'ภูมิใจไทย',
 'พลังธรรมใหม่',
 'กรีน',
 'ไทยธรรม',
 'แผ่นดินธรรม',
 'กล้าธรรม',
 'พลังประชารัฐ',
 'โอกาสใหม่',
 'เป็นธรรม',
 'ประชาชน',
 'ประชาไทย',
 'ไทยสร้างไทย',
 'ไทยก้าวใหม่',
 'ประชาอาสาชาติ',
 'พร้อม',
 'เครือข่ายชาวนาแห่งประเทศไทย',
 'ไทยพิทักษ์ธรรม',
 'ความหวั

In [839]:
df_test = df.copy()

In [840]:
wrong_names = []
for wrong_name in df_test['name'].unique().tolist():
    if wrong_name not in all_subdistrict:
        wrong_names.append(wrong_name)

wrong_names

['ไทยทรัพย์ทวี',
 'เพื่อชาติไทย',
 'ใหม่',
 'มิติใหม่',
 'รวมใจไทย',
 'รวมไทยสร้างชาติ',
 'พลวัต',
 'ประชาธิปไตยใหม่',
 'เพื่อไทย',
 'ทางเลือกใหม่',
 'เศรษฐกิจ',
 'เสรีรวมไทย',
 'รวมพลังประชาชน',
 'ท้องที่ไทย',
 'อนาคตไทย',
 'พลังเพื่อไทย',
 'ไทยชนะ',
 'พลังสังคมใหม่',
 'สังคมประชาธิปไตย',
 'ฟิวชัน',
 'ไทรวมพลัง',
 'ก้าวอิสระ',
 'ปวงชนไทย',
 'วิชชั่นใหม่',
 'เพื่อชีวิตใหม่',
 'คลองไทย',
 'ประชาธิปัตย์',
 'ไทยก้าวหน้า',
 'ไทยภักดี',
 'แรงงานสร้างชาติ',
 'ประชากรไทย',
 'ครูไทยเพื่อประชาชน',
 'ประชาชาติ',
 'สร้างอนาคตไทย',
 'รักชาติ',
 'ไทยพร้อม',
 'ภูมิใจไทย',
 'พลังธรรมใหม่',
 'กรีน',
 'ไทยธรรม',
 'แผ่นดินธรรม',
 'กล้าธรรม',
 'พลังประชารัฐ',
 'โอกาสใหม่',
 'เป็นธรรม',
 'ประชาชน',
 'ประชาไทย',
 'ไทยสร้างไทย',
 'ไทยก้าวใหม่',
 'ประชาอาสาชาติ',
 'พร้อม',
 'เครือข่ายชาวนาแห่งประเทศไทย',
 'ไทยพิทักษ์ธรรม',
 'ความหวังใหม่',
 'ไทยรวมไทย',
 'เพื่อบ้านเมือง',
 'พลังไทยรักชาติ',
 'นางสาววิชุดา ว่องวัฒนวิโรจน์',
 'นางสาวสุวิภา กุศลจูง',
 'นายศรีพรหม หอมยก',
 'นายสมบูรณ์ รูปสะอาด',
 'นายดาชัย เอกปฐ

In [841]:
wrong2correct, scores, list_wrongs = automate_edit('name', correct_names, wrong_names, df_test)

OCR: 'ไทยทรัพย์ทวี' -> Corrected: 'ไทยทรัพย์ทวี' (Confidence: 100.00%)
OCR: 'เพื่อชาติไทย' -> Corrected: 'เพื่อชาติไทย' (Confidence: 100.00%)
OCR: 'ใหม่' -> Corrected: 'ใหม่' (Confidence: 100.00%)
OCR: 'มิติใหม่' -> Corrected: 'มิติใหม่' (Confidence: 100.00%)
OCR: 'รวมใจไทย' -> Corrected: 'รวมใจไทย' (Confidence: 100.00%)
OCR: 'รวมไทยสร้างชาติ' -> Corrected: 'รวมไทยสร้างชาติ' (Confidence: 100.00%)
OCR: 'พลวัต' -> Corrected: 'พลวัต' (Confidence: 100.00%)
OCR: 'ประชาธิปไตยใหม่' -> Corrected: 'ประชาธิปไตยใหม่' (Confidence: 100.00%)
OCR: 'เพื่อไทย' -> Corrected: 'เพื่อไทย' (Confidence: 100.00%)
OCR: 'ทางเลือกใหม่' -> Corrected: 'ทางเลือกใหม่' (Confidence: 100.00%)
OCR: 'เศรษฐกิจ' -> Corrected: 'เศรษฐกิจ' (Confidence: 100.00%)
OCR: 'เสรีรวมไทย' -> Corrected: 'เสรีรวมไทย' (Confidence: 100.00%)
OCR: 'รวมพลังประชาชน' -> Corrected: 'รวมพลังประชาชน' (Confidence: 100.00%)
OCR: 'ท้องที่ไทย' -> Corrected: 'ท้องที่ไทย' (Confidence: 100.00%)
OCR: 'อนาคตไทย' -> Corrected: 'อนาคตไทย' (Confidence: 100.00

In [842]:
list_wrongs

[]

In [843]:
wrong2correct = {
    "พิวซัน": 'ฟิวชัน',
    "ก้าวล่วง": 'ก้าวอิสระ',
    "พิวจัน":"ฟิวชัน",
    "พิกุลัน":"ฟิวชัน",
    "พิวจันทร์":"ฟิวชัน",
    "พิวซิน":"ฟิวชัน",
    "พิวขัน":"ฟิวชัน",
    "กษิน":"กรีน",
    "วิชัยขับใหญ่":"วิชชั่นใหม่",
    "พิชัย":"ฟิวชัน",
    "ทรอม":"พร้อม",
    "พืชขัน":"ฟิวชัน",
    "พืชชัน":"ฟิวชัน",
    "วิภาวดี":"รักชาติ",
    "กรุงเทพมหานคร":"กรีน",
    "ไทยทรัพยากร":"ไทยทรัพย์ทวี",
    "พลังประจวบ":"พลังประชารัฐ",
    "พิวัฒน์":"ฟิวชัน",
    "กวิน":"กรีน",
    "ดร.ณฐภัทร":"เศรษฐกิจ",
    "ศรีราชาไทย":"เสรีรวมไทย",
    "พิมพ์ขึ้น":"ฟิวชัน", 
    "โทรสารเคล็ง":"ไทรวมพลัง",
    "กำกวติสร":"ก้าวอิสระ",
    "พิกุลชน":"ฟิวชัน",
    "นางสาววิฤดา วงศ์พรมไพรณ์":correct_names[0],
    "พิกวัน":"ฟิวชัน",
    "พืชซัน":"ฟิวชัน",
    "พืชชน":"ฟิวชัน",
    "ฟิลิปปิน":"ฟิวชัน",
    "ไทกรุงเทพฯ":"ไทรวมพลัง",
    "ก้าวก้มลง":"ก้าวอิสระ",
    "พิกขัน":"ฟิวชัน",
    "พิกุล":"ฟิวชัน",
    "กำกวติสระ":"ก้าวอิสระ"
}

In [844]:
for wrong in list_wrongs:
    print(df_test[df_test['name'] == wrong])

In [845]:
for wrong, correct in wrong2correct.items():
    df_test.loc[df_test['name'] == wrong, 'name'] = correct

In [846]:
for wrong in list_wrongs:
    print(df_test[df_test['name'] == wrong])

In [847]:
for i in df_test['name'].unique().tolist():
    if i not in correct_names:
        print(i)

In [848]:
df_test.shape

(21496, 7)

In [849]:
df = df_test.copy()

In [850]:
def check_from_df(df):
    expected_khet = set(correct_names[:7])
    expected_bch = set(correct_names[7:])
    
    df['name'] = df['name'].astype(str).str.strip()
    
    print("🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...\n")
    has_error = False
    
    # 3. จัดกลุ่มตรวจสอบ (เพิ่ม 'district' เข้าไปในวงเล็บ)
    grouped = df.groupby(['district', 'sub-district', 'unit_number', 'type'])
    total = 0
    
    # เพิ่ม district มารับค่าในลูปด้วย
    for (district, sub_district, unit_number, type_val), group_data in grouped:
        # ดึงชื่อออกมาเป็น List (เพื่อนับซ้ำ) และ Set (เพื่อหาตัวที่ขาด/เกิน)
        actual_names_list = group_data['name'].tolist()
        actual_names_set = set(actual_names_list)
        
        if type_val == 'เขต':
            expected_names = expected_khet
        elif type_val == 'บช':
            expected_names = expected_bch
        else:
            continue 
        
        expected_total = len(expected_names)
    
        # --- เริ่มกระบวนการตรวจสอบ 3 ชั้น ---
        
        # 1. เช็คขาด (Missing)
        missing = expected_names - actual_names_set
        
        # 2. เช็คแปลกปลอม (Extra/Unrecognized)
        extra = actual_names_set - expected_names
        
        # 3. เช็คซ้ำซ้อน (Duplicates)
        name_counts = group_data['name'].value_counts()
        duplicates = name_counts[name_counts > 1].index.tolist()
    
        # ถ้าเจอความผิดปกติอย่างใดอย่างหนึ่ง ให้ปริ้นท์แจ้งเตือน
        if missing or extra or duplicates:
            has_error = True
            
            # 📌 แก้ไขบรรทัด Print ให้แสดง "อำเภอ" ด้วย
            print(f"📌 [{type_val}] อำเภอ: {district} | ตำบล: {sub_district} | หน่วยที่: {unit_number}")
            print(f"   📊 มีข้อมูลทั้งหมด {len(actual_names_list)} บรรทัด (จากที่ควรมี {expected_total} บรรทัด)")
            
            if missing:
                print(f"   ❌ ขาดหายไป {len(missing)} รายชื่อ: {', '.join(sorted(missing))}")
                total += len(missing)
            if extra:
                print(f"   🚨 แปลกปลอม/ผิดสเปก {len(extra)} รายชื่อ: {', '.join(sorted(extra))}")
                
            if duplicates:
                print(f"   🔄 รายชื่อซ้ำซ้อน {len(duplicates)} รายการ:")
                for dup in duplicates:
                    print(f"      - '{dup}' โผล่มา {name_counts[dup]} บรรทัด")
                    
            print("-" * 60)
            
    print(f"จำนวนที่ขาดรวมคิดเป็น: {total}")
    if not has_error:
        print("✅ ข้อมูลสมบูรณ์แบบ! ทุกหน่วยมีรายชื่อครบถ้วน ไม่มีขาด ไม่มีเกิน ไม่มีซ้ำครับ")

In [851]:
def edit_df(previous_val, old_val, new_val, df):

    previous_name = df.groupby(['district', 'sub-district', 'unit_number', 'type'])['name'].shift(1)

    condition_fix = (df['name'] == old_val) & (previous_name == previous_val)

    rows_to_fix = condition_fix.sum()

    if rows_to_fix > 0:
        df.loc[condition_fix, 'name'] = new_val
        print(f"✅ แก้ไขพรรค {old_val} เป็น {new_val} (ที่อยู่ต่อจาก {previous_val}) สำเร็จจำนวน {rows_to_fix} แถว")
    else:
        print(f"✅ ไม่พบเคสที่พรรค {old_val} อยู่ต่อจาก {previous_val}")

In [852]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [853]:
edit_df("ปวงชนไทย","ใหม่", "วิชชั่นใหม่", df)

✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก ปวงชนไทย


In [854]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [855]:
edit_df("ไทยสร้างไทย", 'ใหม่', "ไทยก้าวใหม่", df)

✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก ไทยสร้างไทย


In [856]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [857]:
edit_df("พลวัต", "ใหม่", "ประชาธิปไตยใหม่", df)

✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก พลวัต


In [858]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [859]:
df.shape

(21496, 7)

In [860]:
print(df[(df['sub-district'] == 'ทุ่งฮั้ว') & (df['unit_number'] == 7)].to_string())

     type province  district sub-district  unit_number                          name score
6366  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7  นางสาววิชุดา ว่องวัฒนวิโรจน์    21
6367  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7          นางสาวสุวิภา กุศลจูง    57
6368  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7              นายศรีพรหม หอมยก     3
6369  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7           นายสมบูรณ์ รูปสะอาด     3
6370  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7              นายดาชัย เอกปฐพี    12
6371  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7           นายธนาธร โล่ห์สุนทร    45
6372  เขต    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7    พันเอกสันทัด ภัทรกิตตินนท์     2
7321   บช    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7                  ไทยทรัพย์ทวี     3
7322   บช    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7                  เพื่อชาติไทย     8
7323   บช    ลำปาง  วังเหนือ     ทุ่งฮั้ว            7                          ใหม่     0

In [861]:
edit_df("ก้าวอิสระ", "ประชาไทย", "ปวงชนไทย", df)
edit_df("ปวงชนไทย", "ใหม่", "วิชชั่นใหม่", df)
check_from_df(df)

✅ ไม่พบเคสที่พรรค ประชาไทย อยู่ต่อจาก ก้าวอิสระ
✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก ปวงชนไทย
🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรั

In [862]:
print(df[(df['sub-district'] == 'บ้านสา') & (df['unit_number'] == 1)].to_string())

      type province district sub-district  unit_number                          name score
17081  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1  นางสาววิชุดา ว่องวัฒนวิโรจน์    13
17082  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1          นางสาวสุวิภา กุศลจูง    99
17083  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1              นายศรีพรหม หอมยก    11
17084  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1           นายสมบูรณ์ รูปสะอาด     8
17085  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1              นายดาชัย เอกปฐพี    10
17086  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1           นายธนาธร โล่ห์สุนทร    54
17087  เขต    ลำปาง   แจ้ห่ม       บ้านสา            1    พันเอกสันทัด ภัทรกิตตินนท์    11
17529   บช    ลำปาง   แจ้ห่ม       บ้านสา            1                  ไทยทรัพย์ทวี     0
17530   บช    ลำปาง   แจ้ห่ม       บ้านสา            1                  เพื่อชาติไทย     9
17531   บช    ลำปาง   แจ้ห่ม       บ้านสา            1                          ใหม่     1

In [863]:
edit_df("วิชชั่นใหม่", "เพื่อชาติไทย", "เพื่อชีวิตใหม่", df)

✅ ไม่พบเคสที่พรรค เพื่อชาติไทย อยู่ต่อจาก วิชชั่นใหม่


In [864]:
print(df[(df['sub-district'] == 'วังทอง') & (df['unit_number'] == 9)].to_string())

      type province  district sub-district  unit_number                          name score
6730   เขต    ลำปาง  วังเหนือ       วังทอง            9  นางสาววิชุดา ว่องวัฒนวิโรจน์     7
6731   เขต    ลำปาง  วังเหนือ       วังทอง            9          นางสาวสุวิภา กุศลจูง    48
6732   เขต    ลำปาง  วังเหนือ       วังทอง            9              นายศรีพรหม หอมยก     3
6733   เขต    ลำปาง  วังเหนือ       วังทอง            9           นายสมบูรณ์ รูปสะอาด     0
6734   เขต    ลำปาง  วังเหนือ       วังทอง            9              นายดาชัย เอกปฐพี   142
6735   เขต    ลำปาง  วังเหนือ       วังทอง            9           นายธนาธร โล่ห์สุนทร    26
6736   เขต    ลำปาง  วังเหนือ       วังทอง            9    พันเอกสันทัด ภัทรกิตตินนท์     4
10172   บช    ลำปาง  วังเหนือ       วังทอง            9                  ไทยทรัพย์ทวี     ☑
10173   บช    ลำปาง  วังเหนือ       วังทอง            9                  เพื่อชาติไทย     ☐
10174   บช    ลำปาง  วังเหนือ       วังทอง            9                         

In [865]:
edit_df("ไทยทรัพย์ทวี", "ใหม่", "เพื่อชาติไทย", df)
edit_df("ใหม่", "ใหม่", "มิติใหม่", df)

✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก ไทยทรัพย์ทวี


✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก ใหม่


In [866]:
print(df[(df['sub-district'] == 'เมืองมาย') & (df['unit_number'] == 1)].to_string())

      type province district sub-district  unit_number                          name score
17235  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1  นางสาววิชุดา ว่องวัฒนวิโรจน์     1
17236  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1          นางสาวสุวิภา กุศลจูง    19
17237  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1              นายศรีพรหม หอมยก     1
17238  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1           นายสมบูรณ์ รูปสะอาด     1
17239  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1              นายดาชัย เอกปฐพี   109
17240  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1           นายธนาธร โล่ห์สุนทร    25
17241  เขต    ลำปาง   แจ้ห่ม     เมืองมาย            1    พันเอกสันทัด ภัทรกิตตินนท์     1
19266   บช    ลำปาง   แจ้ห่ม     เมืองมาย            1                      เศรษฐกิจ     5
19267   บช    ลำปาง   แจ้ห่ม     เมืองมาย            1                    เสรีรวมไทย     4
19268   บช    ลำปาง   แจ้ห่ม     เมืองมาย            1                รวมพลังประชาชน     6

In [867]:
edit_df("คลองไทย", "ประชาชน", "ประชาธิปัตย์", df)

✅ ไม่พบเคสที่พรรค ประชาชน อยู่ต่อจาก คลองไทย


In [868]:
edit_df("คลองไทย", "ประชาชาติ", "ประชาธิปัตย์", df)

✅ ไม่พบเคสที่พรรค ประชาชาติ อยู่ต่อจาก คลองไทย


In [869]:
print(df[(df['sub-district'] == 'แจ้ห่ม(นอกเขต)') & (df['unit_number'] == 7)].to_string())

      type province district    sub-district  unit_number                          name score
17452  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7  นางสาววิชุดา ว่องวัฒนวิโรจน์     9
17453  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7          นางสาวสุวิภา กุศลจูง   103
17454  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7              นายศรีพรหม หอมยก     4
17455  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7           นายสมบูรณ์ รูปสะอาด     9
17456  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7              นายดาชัย เอกปฐพี    80
17457  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7           นายธนาธร โล่ห์สุนทร    45
17458  เขต    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7    พันเอกสันทัด ภัทรกิตตินนท์     4
19977   บช    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7                  ไทยทรัพย์ทวี     3
19978   บช    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7                  เพื่อชาติไทย     9
19979   บช    ลำปาง   แจ้ห่ม  แจ้ห่ม(นอกเขต)            7   

In [870]:
edit_df("สังคมประชาธิปไตย", "ใหม่", "ฟิวชัน", df)

✅ ไม่พบเคสที่พรรค ใหม่ อยู่ต่อจาก สังคมประชาธิปไตย


In [871]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [872]:
total = 0
for district in d:
    for sd, val in d[district].items():
        total += val

In [873]:
total

355

In [874]:
df['sub-district'] = df['sub-district'].astype(str).str.strip()
df['unit_number'] = pd.to_numeric(df['unit_number'], errors='coerce')

print("🔍 ตามล่าหาข้อมูลที่หายไป 'ทั้งก้อน'...")

missing_whole_groups = 0

# 1. เอา Group keys ที่มีอยู่จริงใน df มาเก็บไว้ใน Set (เพื่อให้ค้นหาได้เร็วๆ)
# หน้าตาข้อมูลใน Set จะเป็นแบบ: ('นาแก', 1, 'เขต'), ('นาแก', 1, 'บช')
existing_groups = set(tuple(x) for x in df[['sub-district', 'unit_number', 'type']].dropna().drop_duplicates().values)

# 2. วนลูปตามโครงสร้างความจริงจาก Dictionary d
for district in d:
    for sd, total_units in d[district].items():
        for unit in range(1, total_units + 1): # วนลูปหน่วยที่ 1 ถึง N
            c_unit = unit
            if district == 'นอกเขต':
                c_unit = -1
            # print(sd,c_unit)
            # เช็คว่า 'เขต' ของหน่วยนี้มีใน df ไหม?
            if (sd, c_unit, 'เขต') not in existing_groups:
                missing_whole_groups += 7
                print(f"🚨 หายไปทั้งหน้า! [เขต] ตำบล: {sd} | หน่วยที่: {c_unit} (หายไป 7 รายชื่อ)")
                
            # เช็คว่า 'บช' ของหน่วยนี้มีใน df ไหม?
            if (sd, c_unit, 'บช') not in existing_groups:
                missing_whole_groups += 57
                print(f"🚨 หายไปทั้งหน้า! [บช]  ตำบล: {sd} | หน่วยที่: {c_unit} (หายไป 57 รายชื่อ)")

print("-" * 50)
print(f"✅ จำนวนรายชื่อที่หายไปเพราะหายทั้งก้อน (ไม่ได้เข้า groupby): {missing_whole_groups}")

🔍 ตามล่าหาข้อมูลที่หายไป 'ทั้งก้อน'...
🚨 หายไปทั้งหน้า! [เขต] ตำบล: บ้านเสด็จ | หน่วยที่: 19 (หายไป 7 รายชื่อ)
🚨 หายไปทั้งหน้า! [เขต] ตำบล: บ้านโป่ง | หน่วยที่: 9 (หายไป 7 รายชื่อ)
🚨 หายไปทั้งหน้า! [บช]  ตำบล: บ้านโป่ง | หน่วยที่: 9 (หายไป 57 รายชื่อ)
🚨 หายไปทั้งหน้า! [เขต] ตำบล: บ้านโป่ง | หน่วยที่: 11 (หายไป 7 รายชื่อ)
🚨 หายไปทั้งหน้า! [บช]  ตำบล: ปงเตา | หน่วยที่: 4 (หายไป 57 รายชื่อ)
🚨 หายไปทั้งหน้า! [เขต] ตำบล: ทุ่งฮั้ว | หน่วยที่: 10 (หายไป 7 รายชื่อ)
--------------------------------------------------
✅ จำนวนรายชื่อที่หายไปเพราะหายทั้งก้อน (ไม่ได้เข้า groupby): 142


In [875]:
check_from_df(df)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบ

In [876]:
# df.to_csv("master_result1.csv", index=False)

In [877]:
df.isnull().sum()

type              0
province          0
district          0
sub-district      0
unit_number       0
name              0
score           274
dtype: int64

In [892]:
df_result2 = pd.read_csv("master_result2.csv")
df_result2.head()

,type,province,district,sub-district,unit_number,name,score
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ไทยทรัพย์ทวี,0
1,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,เพื่อชาติไทย,6
2,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ใหม่,4
3,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,มิติใหม่,0
4,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,รวมใจไทย,1


In [893]:
check_from_df(df_result2)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: วังเหนือ | ตำบล: ทุ่งฮั้ว | หน่วยที่: 10
   📊 มีข้อมูลทั้งหมด 35 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 22 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ท้องที่ไทย, ประชากรไทย, ประชาธิปัตย์, ปวงชนไทย, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, รวมพลังประชาชน, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชีวิตใหม่, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ไทยก้าวหน้า, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: ทุ่งฮั้ว | หน่วยที่: 11
   📊 มีข้อมูลทั้งหมด 49 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 8 รายชื่อ: ทางเลือกใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังใต้ | หน่วยที่: 2
   📊 มีข้อมูลทั้งหมด 37 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 20 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาอาสาช

In [894]:
def check_missing_allrecord(df):
    df['sub-district'] = df['sub-district'].astype(str).str.strip()
    df['unit_number'] = pd.to_numeric(df['unit_number'], errors='coerce')
    
    print("🔍 ตามล่าหาข้อมูลที่หายไป 'ทั้งก้อน'...")
    
    missing_whole_groups = 0
    
    # 1. เอา Group keys ที่มีอยู่จริงใน df มาเก็บไว้ใน Set (เพื่อให้ค้นหาได้เร็วๆ)
    # หน้าตาข้อมูลใน Set จะเป็นแบบ: ('นาแก', 1, 'เขต'), ('นาแก', 1, 'บช')
    existing_groups = set(tuple(x) for x in df[['sub-district', 'unit_number', 'type']].dropna().drop_duplicates().values)
    
    # 2. วนลูปตามโครงสร้างความจริงจาก Dictionary d
    for district in d:
        for sd, total_units in d[district].items():
            for unit in range(1, total_units + 1): # วนลูปหน่วยที่ 1 ถึง N
                c_unit = unit
                if district == 'นอกเขต':
                    c_unit = -1
                # print(sd,c_unit)
                # เช็คว่า 'เขต' ของหน่วยนี้มีใน df ไหม?
                if (sd, c_unit, 'เขต') not in existing_groups:
                    missing_whole_groups += 7
                    print(f"🚨 หายไปทั้งหน้า! [เขต] ตำบล: {sd} | หน่วยที่: {c_unit} (หายไป 7 รายชื่อ)")
                    
                # เช็คว่า 'บช' ของหน่วยนี้มีใน df ไหม?
                if (sd, c_unit, 'บช') not in existing_groups:
                    missing_whole_groups += 57
                    print(f"🚨 หายไปทั้งหน้า! [บช]  ตำบล: {sd} | หน่วยที่: {c_unit} (หายไป 57 รายชื่อ)")
    
    print("-" * 50)
    print(f"✅ จำนวนรายชื่อที่หายไปเพราะหายทั้งก้อน (ไม่ได้เข้า groupby): {missing_whole_groups}")

In [895]:
check_missing_allrecord(df_result2)

🔍 ตามล่าหาข้อมูลที่หายไป 'ทั้งก้อน'...
🚨 หายไปทั้งหน้า! [เขต] ตำบล: บ้านเสด็จ | หน่วยที่: 19 (หายไป 7 รายชื่อ)
🚨 หายไปทั้งหน้า! [บช]  ตำบล: ปงเตา | หน่วยที่: 4 (หายไป 57 รายชื่อ)
🚨 หายไปทั้งหน้า! [เขต] ตำบล: ทุ่งฮั้ว | หน่วยที่: 10 (หายไป 7 รายชื่อ)
--------------------------------------------------
✅ จำนวนรายชื่อที่หายไปเพราะหายทั้งก้อน (ไม่ได้เข้า groupby): 71


Mannual above file

In [898]:
df_result2_again = pd.read_csv("master_result2.csv")

In [899]:
check_from_df(df_result2_again)

🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

จำนวนที่ขาดรวมคิดเป็น: 0
✅ ข้อมูลสมบูรณ์แบบ! ทุกหน่วยมีรายชื่อครบถ้วน ไม่มีขาด ไม่มีเกิน ไม่มีซ้ำครับ


In [900]:
df_result2_again.isnull().sum()

type              0
province          0
district          0
sub-district      0
unit_number       0
name              0
score           274
dtype: int64